--------------------------------------------------------------------------------------------------------------------

**This code uses the following datasets:**

    - SHELDUS event-level dataset (csv) from Arizona State University's Center for Emergency Management and Homeland Security:
      https://cemhs.asu.edu/sheldus/metadata#about-sheldus 

    - TIGER/line county shape files from Data.gov:
      https://catalog.data.gov/dataset/tiger-line-shapefile-2021-nation-u-s-counties-and-equivalent-entities 
      
Seen specifically in cell 2, inspiration was taken from jtmaze on GitHub to properly handle and format the larger SHELDUS data set. His work can be found here: https://github.com/jtmaze/climate-claims-SC/blob/main/scripts/SHELDUS.ipynb

--------------------------------------------------------------------------------------------------------------------

## SHELDUS Winter Weather Property Damage Analysis (1995-2023)
The goal of this notebook is to analyze winter weather related property damage in the Midwest at a county-level from 1995-2023 (1995 as that is the start year for the downloaded ERA5 data). This data set will be used alongside a Snowfall Accumulation plot and the following plotted ERA5 variables (for now): geopotential heights and wind speeds for jet streak location, temperature, potential vorticity, and a 2PVU surface, and mean sea-level surface pressure with integrated water vapor transport. The use of the following code is to identify the most susceptible counties/regions based on total property damage, as well as event-specific events (hail, wind, winter weather, etc.), and compare the results to different ERA5 variables to identify any key atmospheric drivers.

## 0.) Import Needed Functions and Make Initial Fixes to the Dataset

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import LogNorm
from pathlib import Path

In [ ]:
# read in the  SHELDUS data
data = pd.read_csv('direct_loss_aggregated_output_25562.csv', index_col = False)

# get rid of data with 0 property damage
data = data[data['PropertyDmg'] > 0]

# some of the column names have bad syntax
data.rename(columns={' Hazard': 'Hazard', ' CountyName': 'CountyName'}, inplace=True)

# take a look at the different columns, and what dtypes they are
for col, dtype in data.dtypes.items():
    print(f"{col}'s dtype: {dtype}")

## 1.) Filtering the data to only include data in the Midwest, in DJF months, with winter weather hazards:
The full SHELDUS dataset covers all U.S. counties and all hazard types. Here we narrow to:
- **Hazard types:** Winter Weather, Hail, Wind, Severe Storm/Thunderstorm
- **Months:** December, January, February
- **Region:** 12-state U.S. Midwest


Other available hazards:
- Flooding, Lightning, Coastal, Tornado, Wildfire, Drought, Heat, Hurricane/Tropical Storm, Fog, Avalanche, Landslide, Earthquake, Tsunami/Seiche, and Volcano.

In [ ]:
# filter to get only winter weather in preferred months
winterwx = data.where(data.Hazard == 'Winter Weather')
winterwx = winterwx[winterwx['Month'].isin([12, 1, 2])]
winterwx = winterwx.dropna()

# filter to get only hail in preferred months
hailwx = data.where(data.Hazard == 'Hail')
hailwx = hailwx[hailwx['Month'].isin([12, 1, 2])]
hailwx = hailwx.dropna()

# filter to get only wind in preferred months
windwx = data.where(data.Hazard == 'Wind')
windwx = windwx[windwx['Month'].isin([12, 1, 2])]
windwx = windwx.dropna()

# filter to get only severe storm/thunder storm in preferred months
stormwx = data.where(data.Hazard == 'Severe Storm/Thunder Storm')
stormwx = stormwx[stormwx['Month'].isin([12, 1, 2])]
stormwx = stormwx.dropna()

# since we are only focusing on the Midwest
midwest_states = [
    'ILLINOIS', 'INDIANA', 'IOWA', 'KANSAS', 'MICHIGAN', 'MINNESOTA',
    'MISSOURI', 'NEBRASKA', 'NORTH DAKOTA', 'OHIO', 'SOUTH DAKOTA', 'WISCONSIN']
winterMW = winterwx[winterwx['StateName'].isin(midwest_states)]
winterMW = winterMW.dropna()
hailMW = hailwx[hailwx['StateName'].isin(midwest_states)]
hailMW = hailMW.dropna()
windMW = windwx[windwx['StateName'].isin(midwest_states)]
windMW = windMW.dropna()
stormMW = stormwx[stormwx['StateName'].isin(midwest_states)]
stormMW = stormMW.dropna()

print(f"Winter Weather events: {winterMW['Hazard'].shape[0]}") 
print(f"\nHail events: {hailMW['Hazard'].shape[0]}") 
print(f"\nWind events: {windMW['Hazard'].shape[0]}") 
print(f"\nSevere Storm/Thunder Storm events: {stormMW['Hazard'].shape[0]}") 

## 2.) Building the Combined Dataset
All four hazard types are merged into a single Midwest DataFrame for aggregate spatial analysis. The dataset is split at 1995 to align with ERA5 reanalysis availability:

- **`dataMWpre`**: pre-1995 events (kept for comparison plot)
- **`dataMW`**: 1995–present (where the majority of analysis will be done)

In [ ]:
# only Midwest data, in DJF months, in selected weather events
dataMW = data[data['StateName'].isin(midwest_states)]
dataMW = dataMW[dataMW['Hazard'].isin(['Winter Weather', 'Hail', 'Wind', 'Severe Storm/Thunder Storm'])]
dataMW = dataMW[dataMW['Month'].isin([12, 1, 2])]
dataMWpre = dataMW[dataMW['Year'] < 1995].copy() # for the comparison plot later on
dataMW = dataMW[dataMW['Year'] >= 1995].copy().dropna() # since our ERA5 data only goes back to 1995

print(f"Midwest DJF event count (1995-2023): {dataMW['Hazard'].shape[0]}") 
print(f"\nTop 10 Costliest Disasters (1995-2023)")
print(f"\n{dataMW.sort_values(by='PropertyDmg(ADJ 2023)', ascending=False).head(10)}")

## 3.) Outlier Removal: Missouri Ice Storm

The single costliest event in the dataset is a January 2001 ice storm in Missouri that deposited up to **1.5 inches of ice**, caused widespread power outages lasting ~3 days, and dwarfs all other events in adjusted property damage. Including it would severely skew any choropleth or time-series visualization. The four SHELDUS rows associated with this event (split across Missouri counties) are removed here and referred to throughout the notebook as **`MOicestorm`**.

In [ ]:
# locating just the top 4 events (in Missouri), and dropping them from the data
bigevents = dataMW.sort_values(by='PropertyDmg(ADJ 2023)', ascending=False)
MOicestorm = bigevents.index[:4]
dataMW = dataMW.drop(MOicestorm)

dataMW.sort_values(by='PropertyDmg(ADJ 2023)', ascending=False)

# what is the distribution of the data now?
dataMW['PropertyDmg(ADJ 2023)'].describe()

## 4.) Removing Low-Damage Events

The output above shows a minimum adjusted damage value near **16 USD**, not very useful for our analysis. The 25th percentile sits around **3500 USD**, meaning a quarter of events contribute fairly negligible damage and would compress the color scale in spatial plots. We are much more worried about the extremely costly events and their relation to winter weather events.

Events below **$3500** are removed from both the primary and pre-1995 datasets.

In [ ]:
DAMAGE_THRESHOLD = 3496 # approximately the 25th percentile of dataMW

dataMW    = dataMW[dataMW['PropertyDmg(ADJ 2023)'] >= DAMAGE_THRESHOLD].dropna()
dataMWpre = dataMWpre[dataMWpre['PropertyDmg(ADJ 2023)'] >= DAMAGE_THRESHOLD].dropna()

print(f"dataMW after $3500 and under removal: {dataMW.shape[0]} events")
print(f"dataMWpre after $3500 and under removal: {dataMWpre.shape[0]} events")

## 5.) County Shapefile Setup & FIPS Code Alignment

TIGER/Line county shapefiles are joined to the SHELDUS loss data using **5-digit FIPS codes** as the common key. A few steps are required before merging:

- The shapefile is filtered to the 12 Midwest states upfront
- SHELDUS `County_FIPS` values are zero-padded to 5 digits (e.g. 1001 to 01001) to match the shapefile's GEOID format
- **left join** retains all Midwest counties, so those with no recorded damage appear as `NaN` and render as white on the map

In [ ]:
# FIPS for the 12 midwest states
midwest_fips = ['17','18','19','20','26','27','29','31','38','39','46','55']

# load the 2021 TIGER/Line county shapefile (download from: https://www.census.gov/cgi-bin/geo/shapefiles/index.php)
# filter to only Midwest counties to reduce memory usage and speed up plotting
counties = gpd.read_file('./countyshapes/tl_2021_us_county.shp') 
counties = counties[counties['STATEFP'].isin(midwest_fips)]

# zero-pad FIPS codes to 5 digits so they match the shapefile's GEOID format
dataMW['County_FIPS'] = dataMW['County_FIPS'].astype(str).str.zfill(5) 
dataMWpre['County_FIPS'] = dataMWpre['County_FIPS'].astype(str).str.zfill(5)

# sum total adjusted property damage per county across all years and hazards (1995-2023)
county_loss = (dataMW.groupby('County_FIPS', as_index=False)['PropertyDmg(ADJ 2023)'].sum())

# ensure GEOID and County_FIPS are both strings (matching Dtypes) before merging so all rows can match
counties['GEOID'] = counties['GEOID'].astype(str)
county_loss['County_FIPS'] = county_loss['County_FIPS'].astype(str).str.zfill(5)

# left join keeps all Midwest counties in the shapefile, even those with no recorded damage events (they will appear as NaN and appear white on the map)
mw_gdf = counties.merge(
    county_loss,
    left_on='GEOID',
    right_on='County_FIPS',
    how='left'
)

# create a Lambert conformal projection object, centered over the US that also matches the Snowfall Accumulation Plot
proj_lcc = ccrs.LambertConformal(
    central_longitude=-99.0,
    central_latitude=35.0,
    standard_parallels=[35.0]
)

## 6.) Figure 1: Property Damage by Hazard Type (DJF, 1995–2023)

Four-panel choropleth showing county-level total adjusted property damage broken out by hazard type. A shared log-normalized color scale (anchored at $3500 to avoid errors on near-zero values) ensures panels are directly comparable.

**Layout notes:**
- `wspace=0` removes the horizontal gap between columns for a tighter look
- `set_aspect('auto')` before plotting allows axes to fill their grid cell; `set_aspect(1)` after plotting corrects the horizontal compression introduced by Cartopy
- The Great Lakes are drawn twice: once filled (`lightblue`) and once as an outline only, so county boundaries don't bleed visibly into Lake Michigan

In [ ]:
# aggregate total adjusted property damage per county, separately for each hazard
# result is a list of GeoDataFrames (agg_gdfs), one per hazard, in the same order as 'hazards'
hazards = ['Winter Weather', 'Wind', 'Severe Storm/Thunder Storm', 'Hail']
agg_gdfs = []
for hazard in hazards:
    subset = dataMW[dataMW['Hazard'] == hazard]
    grouped = subset.groupby('County_FIPS', as_index=False)['PropertyDmg(ADJ 2023)'].sum()
    grouped['County_FIPS'] = grouped['County_FIPS'].astype(str).str.zfill(5)
    counties['GEOID'] = counties['GEOID'].astype(str)
    # left join to retain all Midwest counties, even those with no damage for this hazard
    merged = counties.merge(grouped, left_on='GEOID', right_on='County_FIPS', how='left')
    agg_gdfs.append(merged)

# plot setup
fig = plt.figure(figsize=(12, 8))
gs = gridspec.GridSpec(nrows=2, ncols=2, figure=fig,           # 2 by 2 grid with tight spacing
                      wspace=0.05, hspace=0.1,                    # wspace removes horizontal gap between columns
                      top=0.9, bottom=0.1) # top and bottom give room for suptitle and colorbar respectively
# flatten into a list of axis (top left, top right, bottom left, bottom right)
axs = [fig.add_subplot(gs[j, i], projection=proj_lcc) for j in range(2) for i in range(2)]

titles = ['Winter Weather', 'Wind', 'Severe Storm/Thunder Storm', 'Hail']

# shared color limits across all panels for consistent scaling
# use the same norm across all 4 panels so colors are directly comparable
vmin = county_loss['PropertyDmg(ADJ 2023)'].min()
vmax = county_loss['PropertyDmg(ADJ 2023)'].max()
norm = LogNorm(vmin=3500, vmax=vmax) 

# PLOT
for ax, gdf, title in zip(axs, agg_gdfs, titles):
    # ax.set_extent([-105, -80, 35, 50], crs=ccrs.PlateCarree()) # to include only the zoomed out Midwest states
    ax.set_extent([-122, -72.5, 21.5, 50], crs=ccrs.PlateCarree())  # full CONUS
    ax.set_aspect('auto') # set_aspect('auto') first to allow the axes to fill the grid cell
    
    # add cartopy features
    ax.coastlines(resolution='50m', color='black', linewidths=1.2, zorder = 3) 
    ax.add_feature(cfeature.BORDERS, linewidths=1.2, edgecolor='black', zorder =3)
    state_borders = cfeature.NaturalEarthFeature(category='cultural', name='admin_1_states_provinces_lakes', scale='50m', facecolor='none')
    ax.add_feature(state_borders, edgecolor='black', linewidth=0.5, zorder = 4)
    land_mask = cfeature.NaturalEarthFeature('physical', 'land', '50m', edgecolor='face', facecolor=cfeature.COLORS['land']) # land
    sea_mask = cfeature.NaturalEarthFeature('physical', 'ocean', '50m', edgecolor='face', facecolor=cfeature.COLORS['water']) # ocean
    lake_mask = cfeature.NaturalEarthFeature('physical', 'lakes', '50m', edgecolor='face', facecolor=cfeature.COLORS['water']) # lakes
    ax.add_feature(sea_mask, zorder=0) 
    ax.add_feature(land_mask, zorder=0)
    ax.add_feature(lake_mask, zorder=4)
    ax.add_feature(cfeature.LAKES, edgecolor='black', facecolor='none', linewidth=.75, zorder=5) # outlining over the county lines
    
    gdf.plot(
        column='PropertyDmg(ADJ 2023)',
        cmap='Reds',
        norm = norm,
        linewidth=0.2,
        ax=ax,
        edgecolor='black',
        transform=ccrs.PlateCarree(), # data is in geographic coords, reprojected to proj_lcc
        vmin=vmin,
        vmax=vmax
    )
    ax.set_title(title, fontsize=15)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect(1) # fix horizontal compression
    
    for spine in ax.spines.values(): 
        spine.set_edgecolor('black')
        spine.set_linewidth(1.2)

# shared colorbar
sm = plt.cm.ScalarMappable(cmap='Reds', norm=norm)
cbar_ax = fig.add_axes([0, 0.05, 1, 0.035]) # [left, bottom, width, height]
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', format='%.1e')
cbar.set_label('Property Damage (2023 USD)', fontsize=13)

fig.suptitle("Hazards Across Midwest Counties (DJF, 1995–2023)", fontsize=20, fontweight='bold') # originally 1982-2023

# save
fname = "CountyLossPerEvent.png"
save_path = Path("/home1/alamia/Research/SHELDUS/") / fname
plt.savefig(save_path, dpi=100, bbox_inches='tight')
print(f"Saved to: {os.path.abspath(save_path)}")

plt.show()

## 7.) Figure 2: Total Midwest Property Damage (DJF, 1995–2023)

All four hazard types are combined into a single county-level total damage map. This is the primary figure intended for overlaying. 

Future steps:
- Overlay ERA5 composites on high-damage years
- Overlay NWS winter weather warnings to assess warning adequacy
- Apply findings toward insurance/reinsurance loss modeling

**Note:** This plot uses a **linear** `Normalize` rather than `LogNorm` (used in the 4-panel plot) to emphasize absolute damage magnitude.

In [ ]:
# sum adjusted property damage across all hazards and years for each county
total = dataMW.groupby('County_FIPS', as_index=False)['PropertyDmg(ADJ 2023)'].sum()
total['County_FIPS'] = total['County_FIPS'].astype(str).str.zfill(5)

total_merged = counties.merge(total, left_on='GEOID', right_on='County_FIPS', how='left')

# plot setup
fig, ax = plt.subplots(figsize=(12,8), dpi=125, subplot_kw={'projection': proj_lcc}) # subplot_kw passes the LCC projection to the single axes object
# ax.set_extent([-105, -80, 35, 50], crs=ccrs.PlateCarree()) # to include only the zoomed out Midwest states
ax.set_extent([-122, -72.5, 21.5, 50], crs=ccrs.PlateCarree()) # full CONUS

# LINEAR color scale
vmin = dataMW['PropertyDmg(ADJ 2023)'].min()
vmax = dataMW['PropertyDmg(ADJ 2023)'].max()

ax.coastlines(resolution='50m', color='black', linewidths=1.2, zorder = 3) 
ax.add_feature(cfeature.BORDERS, linewidths=1.2, edgecolor='black', zorder =3)
state_borders = cfeature.NaturalEarthFeature(category='cultural', name='admin_1_states_provinces_lakes', scale='50m', facecolor='none')
ax.add_feature(state_borders, edgecolor='black', linewidth=0.5, zorder = 4)
land_mask = cfeature.NaturalEarthFeature('physical', 'land', '50m', edgecolor='face', facecolor=cfeature.COLORS['land']) # land
sea_mask = cfeature.NaturalEarthFeature('physical', 'ocean', '50m', edgecolor='face', facecolor=cfeature.COLORS['water']) # ocean
lake_mask = cfeature.NaturalEarthFeature('physical', 'lakes', '50m', edgecolor='face', facecolor=cfeature.COLORS['water']) # lakes
ax.add_feature(sea_mask, zorder=0) 
ax.add_feature(land_mask, zorder=0)
ax.add_feature(lake_mask, zorder=4)
ax.add_feature(cfeature.LAKES, edgecolor='black', facecolor='none', linewidth=.75, zorder=5) # outlining over the county lines

# PLOT
total_merged.plot(
    column='PropertyDmg(ADJ 2023)',
    cmap='Reds',
    linewidth=0.2,
    ax=ax,
    edgecolor='black',
    transform=ccrs.PlateCarree(),
    vmin=vmin,
    vmax=vmax
)

ax.set_xticks([])
ax.set_yticks([])
ax.set_aspect(1) 

# colorbar
sm = plt.cm.ScalarMappable(cmap='Reds', norm=plt.Normalize(vmin=vmin, vmax=vmax))
cax = fig.add_axes([0.12, 0.05, 0.76, 0.025]) 
cbar = fig.colorbar(sm, cax=cax, orientation='horizontal', format='%.1e')
cbar.set_label("Property Damage (2023 USD)", fontsize=13)

fig.suptitle("Total Midwest Property Damage (DJF, 1995–2023)", fontsize=20, y=.95, fontweight='bold')

# save
fname = "TotalCountyLoss.png"
save_path = Path("/home1/alamia/Research/SHELDUS/") / fname
plt.savefig(save_path, dpi=100, bbox_inches='tight')
print(f"Saved to: {os.path.abspath(save_path)}")

plt.show()

## 8.) Figure 3: Pre vs. Post 1995 Average Annual Damage (DJF)

Splitting the record at 1995 (downloaded ERA5 cutoff) allows a rough look at whether the damage distribution has shifted. Damage is normalized by the number of years in each period to make the panels comparable despite their unequal lengths (13 years pre-1995, and 28 years post-1995).

- **Note:** The pre-1995 panel appears spatially smoother than the post-1995 panel. This is likely a function of the shorter record length; fewer years means fewer chances to accumulate the extreme outlier events that produce more boldly red shaded counties. Also, SHELDUS distributes event-level damage evenly across all affected counties, meaning that the same event can lead to similarly shaded counties.

In [ ]:
# pair each dataframe with its number of years for per-year normalization
periods = [dataMWpre, dataMW]
year_counts = [13, 28] #1982-1994 and 1995-2023
agg_gdfs = []

# divide total damage by number of years to get avg annual damage per county
for df, nyears in zip(periods, year_counts):
    grouped = df.groupby('County_FIPS', as_index=False)['PropertyDmg(ADJ 2023)'].sum()
    grouped['County_FIPS'] = grouped['County_FIPS'].astype(str).str.zfill(5)
    grouped['PropertyDmg(ADJ 2023)'] = grouped['PropertyDmg(ADJ 2023)'] / nyears  # average per year
    counties['GEOID'] = counties['GEOID'].astype(str)
    merged = counties.merge(grouped, left_on='GEOID', right_on='County_FIPS', how='left')
    agg_gdfs.append(merged)

# plot setup
fig = plt.figure(figsize=(8, 12))
gs = gridspec.GridSpec(nrows=2, ncols=1, figure=fig, # 2 rows, 1 column 
                       wspace=0, hspace=0.1,         # top/bottom margins leave room for suptitle and colorbar respectively
                       top=0.9, bottom=0.1)
axs = [fig.add_subplot(gs[j, 0], projection=proj_lcc) for j in range(2)]
titles = ['Avg. Annual Midwest Property Damage (DJF, 1982-1994)',
          'Avg. Annual Midwest Property Damage (DJF, 1995–2023)']

# shared color limits across both panels
all_vals = pd.concat([agg_gdfs[0]['PropertyDmg(ADJ 2023)'].dropna(), # compute vmin/vmax from both periods combined so colors are directly comparable
                      agg_gdfs[1]['PropertyDmg(ADJ 2023)'].dropna()])
vmin = all_vals.min()
vmax = all_vals.max()
norm = LogNorm(vmin=3500, vmax=vmax)

# PLOT
for ax, gdf, title in zip(axs, agg_gdfs, titles):
    # ax.set_extent([-105, -80, 35, 50], crs=ccrs.PlateCarree()) # to include only the zoomed out Midwest states
    ax.set_extent([-122, -72.5, 21.5, 50], crs=ccrs.PlateCarree())
    ax.set_aspect('auto')
    ax.coastlines(resolution='50m', color='black', linewidths=1.2, zorder = 3) 
    ax.add_feature(cfeature.BORDERS, linewidths=1.2, edgecolor='black', zorder =3)
    state_borders = cfeature.NaturalEarthFeature(category='cultural', name='admin_1_states_provinces_lakes', scale='50m', facecolor='none')
    ax.add_feature(state_borders, edgecolor='black', linewidth=0.5, zorder = 4)
    land_mask = cfeature.NaturalEarthFeature('physical', 'land', '50m', edgecolor='face', facecolor=cfeature.COLORS['land']) # land
    sea_mask = cfeature.NaturalEarthFeature('physical', 'ocean', '50m', edgecolor='face', facecolor=cfeature.COLORS['water']) # ocean
    lake_mask = cfeature.NaturalEarthFeature('physical', 'lakes', '50m', edgecolor='face', facecolor=cfeature.COLORS['water']) # lakes
    ax.add_feature(sea_mask, zorder=0) 
    ax.add_feature(land_mask, zorder=0)
    ax.add_feature(lake_mask, zorder=4)
    ax.add_feature(cfeature.LAKES, edgecolor='black', facecolor='none', linewidth=.75, zorder=5) # outlining over the county lines

    gdf.plot(
        column='PropertyDmg(ADJ 2023)',
        cmap='Reds',
        norm=norm,
        linewidth=0.2,
        ax=ax,
        edgecolor='black',
        transform=ccrs.PlateCarree(),
        vmin=vmin,
        vmax=vmax
    )
    ax.set_title(title, fontsize=15)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect(1)
    
    for spine in ax.spines.values(): # add visible border around each panel
        spine.set_edgecolor('black')
        spine.set_linewidth(1.2)

# shared colorbar
sm = plt.cm.ScalarMappable(cmap='Reds', norm=norm)
cbar_ax = fig.add_axes([0.05, 0.05, 0.9, 0.035])

cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', format='%.1e')
cbar.set_label('Property Damage (2023 USD)', fontsize=13)

fig.suptitle("Avg. Annual Midwest Property Damage: Pre vs. Post 1995 (DJF)", fontsize=20, fontweight='bold', y=.95)

# save
fname = "CountyLossPre1995.png"
save_path = Path("/home1/alamia/Research/SHELDUS/") / fname
plt.savefig(save_path, dpi=100, bbox_inches='tight')
print(f"Saved to: {os.path.abspath(save_path)}")

plt.show()

## 9.) Figure 4: Annual Midwest Winter Property Damage with ENSO Shading (DJF, 1995–2023)

Comparing **inflation-adjusted (2023 USD)** vs. **nominal** annual property damage. Both series are plotted together, with the adjusted line in gray (background context) and the nominal line in black (raw comparison).

Also within this plot are **ENSO phase years** (NOAA CPC Oceanic Nino Index (https://origin.cpc.ncep.noaa.gov/products/analysis_monitoring/ensostuff/ONI_v5.php)):
- **El Nino years** (red bands) — appear to correlate with elevated damage totals, could be a potential focus for future analysis
- **La Nina years** (blue bands)

In [ ]:
# ensure year is integer dtype, it may be read in as float from the CSV
dataMW['Year'] = dataMW['Year'].astype(int)

# annual totals: adjusted and nominal
annual_adj = (
    dataMW
    .groupby('Year')['PropertyDmg(ADJ 2023)']
    .sum()
    .sort_index()
)

annual_unadj = (
    dataMW
    .groupby('Year')['PropertyDmg']
    .sum()
    .sort_index()
)

# ENSO years
el_nino_years = [1983, 1987, 1988, 1992, 1998, 2003, 2010, 2016, 2019] # only years within the 1995–2023 analysis window will be visible on the plot
la_nina_years = [1989, 1999, 2000, 2008, 2011, 2012, 2021, 2022]

# PLOT
fig, ax = plt.subplots(figsize=(10, 3))

# shade each ENSO year as a 1-year wide band (year +- 0.5) behind the data lines
for year in el_nino_years:
    ax.axvspan(year - 0.5, year + 0.5, color='red', alpha=0.15, zorder=0)
for year in la_nina_years:
    ax.axvspan(year - 0.5, year + 0.5, color='blue', alpha=0.15, zorder=0)

# adjusted damage in gray thinner line 
ax.plot(annual_adj.index, annual_adj.values, color='0.6', lw=1, label='Annual Property Damage (ADJ 2023 USD)')

# unadjusted damage in black thicker line 
ax.plot(annual_unadj.index, annual_unadj.values, color='k', lw=2, label='Annual Property Damage (UNADJ)')

ax.set_title('Annual Midwest Winter Property Damage (DJF, 1995–2023)', fontsize=16, fontweight='bold') 
ax.set_xlabel('Year')
ax.set_ylabel('Property Damage (USD)')
ax.set_xlim(annual_adj.index.min(), annual_adj.index.max()) # set x-axis to exactly span the data range with no padding
ax.legend()
plt.show()

## 10.) Figure 5: Annual Property Damage by Hazard Type (DJF, 1995–2023)

Breaking the time series out by hazard type shows the relative contribution of each event category. Key observations:

- **Winter Weather** is the dominant hazard for most of the record, nearly tracking the total damage line on its own
- **Wind** overtakes Winter Weather in at least two years. These two years align with El Nino periods, suggesting that ENSO may modulate the mix of hazard types driving Midwest winter losses, not just the overall damage total, definitely a possible future investigation.

In [ ]:
# assigning each event to a color
hazards = ['Winter Weather', 'Wind', 'Severe Storm/Thunder Storm', 'Hail']

colors = {
    'Winter Weather': 'tab:blue',
    'Wind': 'tab:orange',
    'Severe Storm/Thunder Storm': 'tab:green',
    'Hail': 'tab:red'
}

# annual totals by hazard
annual_by_hazard = {} # build a dict for each hazard type, stored separately so it can be plotted as its own line

for hazard in hazards:
    subset = dataMW[dataMW['Hazard'] == hazard]
    annual = (
        subset
        .groupby('Year')['PropertyDmg(ADJ 2023)']
        .sum()
        .sort_index()
    )
    annual_by_hazard[hazard] = annual

# sum across all hazards combined for the gray total reference line
annual_total = (
    dataMW
    .groupby('Year')['PropertyDmg(ADJ 2023)']
    .sum()
    .sort_index()
)

# PLOT
fig, ax = plt.subplots(figsize=(10, 3))

# total damage line across all hazards
ax.plot(annual_total.index, annual_total.values,
        color='0.6', lw=1, zorder=1, label='Total')

# individual hazard damage totals
for hazard in hazards:
    series = annual_by_hazard[hazard]
    ax.plot(series.index, series.values,
    color=colors[hazard], lw=2, zorder=2, label=hazard)

ax.set_title('Midwest Winter Property Damage by Hazard (DJF, 1995–2023)', fontsize=16, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Property Damage (ADJ 2023 USD)')
# set x-axis to exactly span the data range with no padding
ax.set_xlim(annual_total.index.min(), annual_total.index.max())

ax.legend()
plt.show()